In [1]:
import pandas as pd
import os

In [4]:
# Define the list of combined daily data files you want to process.
# These files should be in the same directory as this script.
combined_files = [
    'Bajaj-Combined-Data.csv', 
    'Eicher-Combined-Data.csv', 
    'Hero-Combined-Data.csv', 
    'Maruti-Combined-Data.csv', 
    'MM-Combined-Data.csv', 
    'Tata-Combined-Data.csv'
]

# Loop through each combined file
for file_name in combined_files:
    print(f"\n--- Processing file: {file_name} ---")
    
    # Check if the file exists
    if not os.path.exists(file_name):
        print(f"Error: The file '{file_name}' was not found. Skipping...")
        continue

    try:
        # Read the combined daily data into a pandas DataFrame.
        # The `thousands=','` argument tells pandas to treat commas in numbers
        # as thousands separators, which should resolve the conversion error.
        combined_df = pd.read_csv(file_name, thousands=',')
        
        # Check the first few rows to ensure the data was read correctly
        print("First 5 rows of the loaded DataFrame:")
        print(combined_df.head())

        # Ensure the 'Date' column is in datetime format and set it as the index
        combined_df['Date'] = pd.to_datetime(combined_df['Date'])
        combined_df.set_index('Date', inplace=True)
        
        # Define the aggregation logic for each column for weekly resampling
        weekly_aggregation = {
            'Open Price': 'first',
            'High Price': 'max',
            'Low Price': 'min',
            'Close Price': 'last',
            'Total Traded Quantity': 'sum',
            'Turnover ₹': 'sum',
            'No. of Trades': 'sum',
            'Deliverable Qty': 'sum',
            'Last Price': 'last',
            'Average Price': 'mean' 
        }
        
        # Resample the data to a weekly frequency ('W') and apply the aggregations
        weekly_df = combined_df.resample('W').agg(weekly_aggregation)
        
        # For simplicity, we'll fill the constant columns like 'Symbol' and 'Series'
        # with the first available value from the daily data.
        if 'Symbol' in combined_df.columns:
            weekly_df['Symbol'] = combined_df['Symbol'].iloc[0]
        if 'Series' in combined_df.columns:
            weekly_df['Series'] = combined_df['Series'].iloc[0]
        
        # Reset the index to make 'Date' a column again before saving
        weekly_df.reset_index(inplace=True)

        # Create a new output file name (e.g., 'Bajaj-Combined-Weekly-Data.csv')
        # by replacing '-Data' with '-Weekly-Data'
        output_file = file_name.replace('-Data.csv', '-Weekly-Data.csv')
        
        # Save the new weekly data to a CSV file
        weekly_df.to_csv(output_file, index=False)
        
        print(f"Successfully converted '{file_name}' to weekly data and saved as '{output_file}'.")
        print(f"Total rows in new weekly file: {len(weekly_df)}")
        
    except Exception as e:
        print(f"An error occurred while processing {file_name}: {e}")

print("\n--- All files have been processed ---")



--- Processing file: Bajaj-Combined-Data.csv ---
First 5 rows of the loaded DataFrame:
       Symbol Series        Date  Prev Close  Open Price  High Price  \
0  BAJAJ-AUTO     EQ  2014-01-01     1910.85      1915.0     1924.40   
1  BAJAJ-AUTO     EQ  2014-01-02     1917.40      1918.0     1928.75   
2  BAJAJ-AUTO     EQ  2014-01-03     1900.05      1893.0     1909.00   
3  BAJAJ-AUTO     EQ  2014-01-06     1895.55      1896.0     1899.90   
4  BAJAJ-AUTO     EQ  2014-01-07     1888.30      1880.0     1899.80   

   Low Price  Last Price  Close Price  Average Price  Total Traded Quantity  \
0    1890.30     1915.05      1917.40        1914.00                  90587   
1    1895.45     1901.00      1900.05        1907.80                 268521   
2    1870.00     1899.55      1895.55        1893.42                 236545   
3    1875.00     1888.00      1888.30        1886.44                 284336   
4    1880.00     1888.00      1889.95        1889.18                 247757   

    